# M3b: Working with Files — in-class notebook

**Files you need (in this folder):** `names.txt`, `nobelPrizes.json`
Sync your fork first so you have them!

**Part 1** walks through the in-class exercise: building a directory of buildings where MACSS classes meet.
**Part 2** is the application exercise: who has won the Nobel Prize?

**Reading:** [CS with Applications §4.1, Working with Files](https://book.cs-apps.org/working_with_data/files/index.html). Each step below links to the section it practices.

**How to use this notebook:**
- **Predict** cells: write your guess in a comment *before* you run the cell.
- **Your turn** cells: fill in `# YOUR CODE HERE`.
- **Check** cells: run them after your code. Silence means it worked; an `AssertionError` means try again.
- **Discuss** prompts: talk it through with your partner.

Remember the common programming pattern:
1. **Load** the data from disk into a data structure
2. **Manipulate** the data
3. **Write** the result (print it, or write it back to disk)

---
# Part 1: Buildings directory

You are working on a project creating a directory of buildings based on where MACSS classes typically meet.
`names.txt` is the university's building list: each building is an abbreviation, a **tab**, then the name and address.

## Step 1: Open, read, close
**Reading:** [§4.1.1 Basic file I/O](https://book.cs-apps.org/working_with_data/files/index.html#basic-file-i-o)

The most basic version: open the file, read the *whole thing* into one string, close it.

- `open()` doesn't read anything yet. It gives you a **file object**, which keeps track of *where you are* in the file.
- `.read()` reads from where you are to the end, and returns one big string.
- `.close()` releases the file. Forgetting it is easy, which is why we'll switch to `with` in Step 2.

In [7]:
f = open("names.txt")
contents = f.read()
f.close()

print(type(contents))
print(contents[:300])

<class 'str'>
Buildings Directory
Abbreviation

Building Name and Address
1155	1155 E 60th St
 A	 Anatomy Building, 1025 E 57th St.
 AB	 Abbott Memorial Hall, 947-51 E. 58th St.
 ACC	 Accelerator Building, 5604-20 S. Ellis Ave.
 AH	 Argonne Cancer Research Hospital, See FMI
 ALAB	 W.C. Allee Laboratory of Animal 


**Predict:** what does `len(contents)` tell us? Is it the number of buildings?

In [8]:
# My prediction:
len(contents)

7673

It's the number of **characters**: `contents` is one string, not a list of lines.

**Predict:** what happens if you call `.read()` twice on the same file object?

In [10]:
# My prediction:
with open("names.txt") as f:
    first = f.read()
    second = f.read()

print(len(first))
print(repr(second))

7673
''


The first `read()` moves the file's position to the end, so the second has nothing left and returns `''` (the reading makes the same point). To read it again, open the file again.

## Step 2: Use `with` and read line by line
**Reading:** [§4.1.1 Basic file I/O](https://book.cs-apps.org/working_with_data/files/index.html#basic-file-i-o)

`with open(...) as f:` closes the file for us when the indented block ends, even if there's an error. From now on, use `with`.

Looping over the file (`for line in f`) gives one line at a time, so you never hold the whole file in memory. That matters for big files.

**Predict:** why is there a blank line between each printed line?

In [11]:
with open("names.txt") as f:
    for i, line in enumerate(f):
        print(line)
        if i == 6:
            break

Buildings Directory

Abbreviation



Building Name and Address

1155	1155 E 60th St

 A	 Anatomy Building, 1025 E 57th St.

 AB	 Abbott Memorial Hall, 947-51 E. 58th St.



Each `line` ends with `"\n"`, and `print` adds *another* newline. `repr()` shows what's *really* in a string: hidden spaces, tabs (`\t`), and newlines (`\n`). It's your best debugging tool for files.

**Discuss:** look at the output below. What separates the abbreviation from the name? What else do you notice at the start of most lines?

In [12]:
with open("names.txt") as f:
    for i, line in enumerate(f):
        print(repr(line))
        if i == 6:
            break

'Buildings Directory\n'
'Abbreviation\n'
'\n'
'Building Name and Address\n'
'1155\t1155 E 60th St\n'
' A\t Anatomy Building, 1025 E 57th St.\n'
' AB\t Abbott Memorial Hall, 947-51 E. 58th St.\n'


## Step 3: Load the data into a data structure

This is step 1 of the pattern: get the data off disk and into a structure you can work with. A list of `[abbreviation, name]` pairs lets us look at each part separately.

**Your turn:** build a list `buildings` where each element is a list `[abbreviation, name_and_address]`.

Hints:
- `line.strip()` removes whitespace (including `\n`) from both ends
- `line.split("\t", 1)` splits on the **first** tab only

In [21]:
buildings = []

# with open("names.txt") as file:
#     next(file)
#     next(file)
#     next(file)
#     next(file)
#     for line in file:
#         line = line.strip()
#         parts = line.split("\t", 1)
#         buildings.append(parts)

with open("names.txt") as file: 
    for i, lines in enumerate(file):
        if i > 4: 
            lines = lines.strip()
            parts = lines.split("\t", 1)
            buildings.append(parts)
print(buildings[:8])

[['A', ' Anatomy Building, 1025 E 57th St.'], ['AB', ' Abbott Memorial Hall, 947-51 E. 58th St.'], ['ACC', ' Accelerator Building, 5604-20 S. Ellis Ave.'], ['AH', ' Argonne Cancer Research Hospital, See FMI'], ['ALAB', ' W.C. Allee Laboratory of Animal Behavior, 940 E. 57th St.'], ['AMB', ' Albert Merritt Billings Hospital, 5841 S. Maryland Ave.'], ['ANL', ' Argonne National Laboratory, 9700 S. Cass Ave., Argonne, IL 60439'], ['ARR', ' To be arranged; contact department or see notes for meeting information']]


In [19]:
# Check
assert len(buildings) == 152, "one entry per line in the file"
assert buildings[4] == ["1155", "1155 E 60th St"]
assert ["SS", " Social Sciences Research Building, 1126 E. 59th St."] in buildings,     "did you strip each line before splitting?"


Look at the first few rows. The file has a header (title, column names, a blank line) that isn't data.
How many lines? Look at the end of the list too.

In [24]:
print(len(buildings))
buildings[-4:]

152


[[''],
 [''],
 [''],
 ['From https://registrar.uchicago.edu/faculty-staff/classroom-scheduling/buildings-directory-2/']]

### What if you want to skip lines?

Two approaches from the slides: call `next()` on the file to skip lines, or use `enumerate` and keep only certain line numbers.

**Discuss:** which approach would still work if the registrar added one more line to the header?

In [25]:
# Approach 1: next() skips one line each time
with open("names.txt") as f:
    next(f)
    next(f)
    next(f)
    next(f)
    first_data_line = f.readline()

repr(first_data_line)

"'1155\\t1155 E 60th St\\n'"

In [9]:
# Approach 2: enumerate. Keep only the lines that are buildings.
# Hint: which line index is the first building? Which is the last?
data_lines = []
# YOUR CODE HERE
with open("names.txt") as f: 
    for i, lines in enumerate(f):
        if i > 3 and i < 148: 
            data_lines.append(lines.strip().split("\t", 1))

print(repr(data_lines[0]))
print(repr(data_lines[-1]))

['1155', '1155 E 60th St']
['ZDC', ' Walter Zoller Memorial Dental Clinic, 5841 S. Maryland Ave.']


In [33]:
# Check
assert data_lines[0].startswith("1155")
assert "ZDC" in data_lines[-1]

## Step 4: Manipulate — select the buildings we need

Step 2 of the pattern: now that the data is in a list, we can filter it. We want these buildings:

In [23]:
sched = ["1155", "SS", "TTI", "K", "STU"]

**Predict:** this looks right. How many of the 5 buildings will it find? Vote: 0, 1, 5, or an error.

In [24]:
# My prediction:
buildings_select = []
with open("names.txt") as f:
    for line in f:
        abbrev = line.split("\t", 1)[0]
        if abbrev in sched:
            buildings_select.append(abbrev)

buildings_select

['1155']

Only one match! Look at `repr()` of the abbreviation for Social Sciences. What's going on?

In [25]:
with open("names.txt") as f:
    for line in f:
        if "Social Sciences" in line:
            print(repr(line))

' SS\t Social Sciences Research Building, 1126 E. 59th St.\n'


Most abbreviations have a **leading space** (`" SS"`), so `" SS" in sched` is `False`. Python compares strings character by character, and a space is a character.

The lesson: **always look at your raw data with `repr()`** before you trust a comparison.

**Your turn:** using your `buildings` list from Step 3, build `buildings_select`: the `[abbrev, name]` pairs whose abbreviation is in `sched`. Do you get all 5?

In [35]:


# with open("names.txt") as f:
#     for line in f:
#         parts = line.split("\t", 1)

#         if len(parts) == 2:
#             abbrev = parts[0].strip()
#             name = parts[1].strip()

#             if abbrev in sched:
#                 buildings_select.append([abbrev, name])

buildings_select = []
with open("names.txt") as f: 
    for i, lines in enumerate(f): 
        if i > 3 and "\t" in lines: 
            parts = lines.split("\t", 1)
            abb = parts[0].strip()
            name = parts[1].strip()
            if abb in sched:
                buildings_select.append([abb, name])

buildings_select

[['1155', '1155 E 60th St'],
 ['K', 'Kent Chemical Laboratory, 1020 E. 58th St.'],
 ['SS', 'Social Sciences Research Building, 1126 E. 59th St.'],
 ['STU', 'Harold Leonard Stuart Hall, 5835 S. Greenwood Ave.'],
 ['TTI', 'Toyota Technical Institute at Chicago, 6045 S. Kenwood Ave.']]

In [36]:
# Check
assert len(buildings_select) == 5
assert [b[0].strip() for b in buildings_select] == ["1155", "K", "SS", "STU", "TTI"]

## Step 5: Write the result to a file
**Reading:** [§4.1.2 Writing data to a file](https://book.cs-apps.org/working_with_data/files/index.html#writing-data-to-a-file)

Step 3 of the pattern. Open the file in **write mode** (`"w"`). `print(..., file=f)` handles the newline for you; `f.write(s)` doesn't, so you'd add `"\n"` yourself.

| mode | what it does |
|---|---|
| `"r"` | read (the default) |
| `"w"` | write: **creates** the file, or **wipes** it if it exists |
| `"a"` | append: adds to the end |

**Predict:** what will `scratch.txt` contain after this cell runs?

In [59]:
# My prediction:
with open("scratch.txt", "w") as f:
    print("first", file=f)

with open("scratch.txt", "w") as f:
    print("second", file=f)

with open("scratch.txt", "a") as f:
    print("third", file=f)

with open("scratch.txt") as f:
    print(f.read())

second
third



Opening in `"w"` wiped `first`. **Be careful:** `"w"` on the wrong file name will erase your data with no warning.

In [38]:
# Write each selected building to selected_buildings.txt, one per line
# YOUR CODE HERE
with open("selected_buildings.txt", "w") as f:
    for abb, name in buildings_select:
        f.write(f"{abb}\t{name}\n")

In [46]:
with open("names.txt", 'r') as f: 
    for i, lines in enumerate(f):
        if i > 4 and i < 148:
            if "\t" in lines:
                part = lines.split("\t", 1)
                abb = part[0]
                name = part[1]
                buildings.append([abb, name])
print(buildings)

[['A', ' Anatomy Building, 1025 E 57th St.'], ['AB', ' Abbott Memorial Hall, 947-51 E. 58th St.'], ['ACC', ' Accelerator Building, 5604-20 S. Ellis Ave.'], ['AH', ' Argonne Cancer Research Hospital, See FMI'], ['ALAB', ' W.C. Allee Laboratory of Animal Behavior, 940 E. 57th St.'], ['AMB', ' Albert Merritt Billings Hospital, 5841 S. Maryland Ave.'], ['ANL', ' Argonne National Laboratory, 9700 S. Cass Ave., Argonne, IL 60439'], ['ARR', ' To be arranged; contact department or see notes for meeting information'], ['ART INT', ' Art Institute of Chicago, 111 S. Michigan Ave. (60603)'], ['B455', ' Booth 455, 455 N Cityfront Plaza Drive'], ['BG', ' Bartlett Commons, 5640 S. University Ave.'], ['BH A', ' Billings, Central Corridor, 5841 S. Maryland Ave.'], ['BH E', ' Billings, East Wing (Armour Clinical Research Building)'], ['BH H', ' Billings, Goldblatt Pavilion'], ['BH M', ' Billings, Medical Corridor'], ['BH N', ' Billings, North Wing'], ['BH O', ' Billings, Orthopedic Corridor'], ['BH P', 

Check your work by reading the file back in:

In [39]:
with open("selected_buildings.txt") as f:
    print(f.read())

1155	1155 E 60th St
K	Kent Chemical Laboratory, 1020 E. 58th St.
SS	Social Sciences Research Building, 1126 E. 59th St.
STU	Harold Leonard Stuart Hall, 5835 S. Greenwood Ave.
TTI	Toyota Technical Institute at Chicago, 6045 S. Kenwood Ave.



Each line is the *printed list* (`['SS', ' Social Sciences...']`). Not very readable!
**Your turn:** write a cleaner version, `selected_buildings_clean.txt`, with lines like `SS: Social Sciences Research Building, 1126 E. 59th St.`

In [41]:
# YOUR CODE HERE
with open("selected_buildings_cleaned.txt", "w") as f:
    for abb, name in buildings_select:
        f.write(f"{abb}: {name}\n")

with open("selected_buildings_cleaned.txt", "r") as f:
    print(f.read())

1155: 1155 E 60th St
K: Kent Chemical Laboratory, 1020 E. 58th St.
SS: Social Sciences Research Building, 1126 E. 59th St.
STU: Harold Leonard Stuart Hall, 5835 S. Greenwood Ave.
TTI: Toyota Technical Institute at Chicago, 6045 S. Kenwood Ave.



## Step 6: Bringing it together in a function
**Reading:** [§4.1.2](https://book.cs-apps.org/working_with_data/files/index.html#writing-data-to-a-file) does the same thing with `strip_domain`: load → transform → write, each as its own block.

Why a function? Next quarter the schedule changes, so you call it with a new `sched`. A new campus file? Pass a new `input_filename`. Nothing inside has to change.

**Your turn:** fill in `get_buildings` so it works for any input file, output file, and schedule.

In [73]:
def get_buildings(input_filename, output_filename, sche):
    '''
    Extract the buildings in sched from the campus list
    and write them to output_filename, one per line.

    Inputs:
      input_filename: (string) name of a file with buildings
      output_filename: (string) name for the output file
      sched: (list of strings) building abbreviations to keep

    Returns: (list of lists) the selected [abbrev, name] pairs
    '''
    # Load data into a data structure (a list of lists)
    buildings = []
    with open(input_filename, 'r') as f: 
        for i, lines in enumerate(f):
            if i > 3 and i < 148:
                if "\t" in lines:
                    part = lines.split("\t", 1)
                    abb = part[0].strip()
                    name = part[1].strip()
                    buildings.append([abb, name])
    print(buildings)

    buildings_select = []
    for abb, name in buildings:
        if abb in sche:
            buildings_select.append([abb, name])

    # Write the data
    with open(output_filename, 'w') as f:
        for abb, name in buildings_select:
            f.write(f"{abb}: \t{name}\n")

    return buildings_select

sche = ["1155", "SS", "TTI", "K", "STU"]
print(get_buildings("names.txt", "new_list.txt", sche))


[['1155', '1155 E 60th St'], ['A', 'Anatomy Building, 1025 E 57th St.'], ['AB', 'Abbott Memorial Hall, 947-51 E. 58th St.'], ['ACC', 'Accelerator Building, 5604-20 S. Ellis Ave.'], ['AH', 'Argonne Cancer Research Hospital, See FMI'], ['ALAB', 'W.C. Allee Laboratory of Animal Behavior, 940 E. 57th St.'], ['AMB', 'Albert Merritt Billings Hospital, 5841 S. Maryland Ave.'], ['ANL', 'Argonne National Laboratory, 9700 S. Cass Ave., Argonne, IL 60439'], ['ARR', 'To be arranged; contact department or see notes for meeting information'], ['ART INT', 'Art Institute of Chicago, 111 S. Michigan Ave. (60603)'], ['B455', 'Booth 455, 455 N Cityfront Plaza Drive'], ['BG', 'Bartlett Commons, 5640 S. University Ave.'], ['BH A', 'Billings, Central Corridor, 5841 S. Maryland Ave.'], ['BH E', 'Billings, East Wing (Armour Clinical Research Building)'], ['BH H', 'Billings, Goldblatt Pavilion'], ['BH M', 'Billings, Medical Corridor'], ['BH N', 'Billings, North Wing'], ['BH O', 'Billings, Orthopedic Corridor']

In [74]:
result = get_buildings("names.txt", "selected_buildings.txt", sche)
assert len(result) == 5, f"expected 5 buildings, got {len(result)}"
result

[['1155', '1155 E 60th St'], ['A', 'Anatomy Building, 1025 E 57th St.'], ['AB', 'Abbott Memorial Hall, 947-51 E. 58th St.'], ['ACC', 'Accelerator Building, 5604-20 S. Ellis Ave.'], ['AH', 'Argonne Cancer Research Hospital, See FMI'], ['ALAB', 'W.C. Allee Laboratory of Animal Behavior, 940 E. 57th St.'], ['AMB', 'Albert Merritt Billings Hospital, 5841 S. Maryland Ave.'], ['ANL', 'Argonne National Laboratory, 9700 S. Cass Ave., Argonne, IL 60439'], ['ARR', 'To be arranged; contact department or see notes for meeting information'], ['ART INT', 'Art Institute of Chicago, 111 S. Michigan Ave. (60603)'], ['B455', 'Booth 455, 455 N Cityfront Plaza Drive'], ['BG', 'Bartlett Commons, 5640 S. University Ave.'], ['BH A', 'Billings, Central Corridor, 5841 S. Maryland Ave.'], ['BH E', 'Billings, East Wing (Armour Clinical Research Building)'], ['BH H', 'Billings, Goldblatt Pavilion'], ['BH M', 'Billings, Medical Corridor'], ['BH N', 'Billings, North Wing'], ['BH O', 'Billings, Orthopedic Corridor']

[['1155', '1155 E 60th St'],
 ['K', 'Kent Chemical Laboratory, 1020 E. 58th St.'],
 ['SS', 'Social Sciences Research Building, 1126 E. 59th St.'],
 ['STU', 'Harold Leonard Stuart Hall, 5835 S. Greenwood Ave.'],
 ['TTI', 'Toyota Technical Institute at Chicago, 6045 S. Kenwood Ave.']]

**Discuss:** in the function, `line[0] in sched` works *without* `.strip()` on the abbreviation. Why? (Hint: what does `line.strip()` do to `" SS\t Social..."`?)

## Exam prep: spot 3 errors

This version of `get_buildings` has **3 errors**. Find them *on paper* first (that's how the exam works), then write them down in the next cell.

For each error, also decide: would Python give you an error message, or would it run and give a wrong answer?

```python
def get_buildings(input_filename, output_filename, sched):
    buildings = []
    with open(input_filename) as f:
        for line in f:
            builds = line.strip().split("\t", 1)
            buildings.append(builds)

    buildings_select = []
    for line[0] in buildings:
        if line in sched:
            buildings_select.append(line)

    with open(output_filename) as f:
        for build in buildings:
            print(build, file=f)
```

In [ ]:
# Your errors:
# 1.
# 2.
# 3.


---
# Part 2: Application — Nobel Prizes

You are a social scientist interested in who has won the Nobel Prize.
`nobelPrizes.json` comes from the Nobel Prize API. It's a tiny slice of the data, so it won't overwhelm your machine.

JSON is how most web APIs send data. It maps neatly onto Python:

| JSON | Python |
|---|---|
| object `{...}` | `dict` |
| array `[...]` | `list` |
| string, number | `str`, `int` / `float` |
| `true` / `false` / `null` | `True` / `False` / `None` |

## Step 1: Load the data
**Reading:** [§4.1.4 JSON](https://book.cs-apps.org/working_with_data/files/index.html#json)

`json.load(f)` reads a JSON **file** into Python data structures.
(`json.loads(s)` does the same from a **string**.)

In [76]:
import json

# Load nobelPrizes.json into a variable called nobel
# YOUR CODE HERE
with open("nobelPrizes.json") as f:
    nobel = json.load(f)


## Step 2: Inspect the data

Before writing any loops, figure out the **shape** of the data. Real data is nested: dicts inside lists inside dicts. If you guess the shape, you'll get `KeyError`s and `TypeError`s. Answer these with code:
1. What type is `nobel`? What are its keys?
2. Where are the prizes? What type is that, and how many are there?
3. What are the keys of one prize?

In [83]:
# YOUR CODE HERE
print(type(nobel))
print(len(nobel))
prizes = nobel["nobelPrizes"]
print(prizes[6]["awardYear"])          # '1901'  (a string, not a number)
print(prizes[4]["category"]["en"])     # 'Chemistry'
print(prizes[0]["laureates"][0]["knownName"]["en"])   # "Jacobus H. van 't Hoff"

<class 'dict'>
3
1902
Physiology or Medicine
Jacobus H. van 't Hoff


Look at one full prize. `json.dumps(..., indent=2)` turns it back into a nicely indented string so you can read it.

**Discuss:** what's the *path* to the year? To the English category? (e.g., `prizes[0]["..."]["..."]`)

In [105]:
prizes = nobel["nobelPrizes"]
print(json.dumps(prizes[9], indent=2))

{
  "awardYear": "1902",
  "category": {
    "en": "Physiology or Medicine",
    "no": "Fysiologi eller medisin",
    "se": "Fysiologi eller medicin"
  },
  "categoryFullName": {
    "en": "The Nobel Prize in Physiology or Medicine",
    "no": "Nobelprisen i fysiologi eller medisin",
    "se": "Nobelpriset i fysiologi eller medicin"
  },
  "dateAwarded": "1902-10-30",
  "prizeAmount": 141847,
  "prizeAmountAdjusted": 10247371,
  "links": [
    {
      "rel": "nobelPrize",
      "href": "https://api.nobelprize.org/2/nobelPrize/med/1902",
      "action": "GET",
      "types": "application/json"
    }
  ],
  "laureates": [
    {
      "id": "294",
      "knownName": {
        "en": "Ronald Ross"
      },
      "fullName": {
        "en": "Ronald Ross"
      },
      "portion": "1",
      "sortOrder": "1",
      "motivation": {
        "en": "for his work on malaria, by which he has shown how it enters the organism and thereby has laid the foundation for successful research on this disease

In [ ]:
# Try the paths you found:


`meta` tells us about the dataset. What does `count` vs. `limit` tell you about how much of the data we have?

In [85]:
nobel["meta"]

{'offset': 0,
 'limit': 200,
 'nobelPrizeYear': 1901,
 'yearTo': 1925,
 'count': 125,
 'terms': 'https://www.nobelprize.org/about/terms-of-use-for-api-nobelprize-org-and-data-nobelprize-org/',
 'license': 'https://www.nobelprize.org/about/terms-of-use-for-api-nobelprize-org-and-data-nobelprize-org/#licence',
 'disclaimer': 'https://www.nobelprize.org/about/terms-of-use-for-api-nobelprize-org-and-data-nobelprize-org/#disclaimer'}

## Step 3: Build a dictionary of years and awards

**Your turn:** build `years_awards`, a dictionary mapping each award year to a list of the categories awarded that year:

```python
{'1901': ['Chemistry', 'Literature', ...], '1902': [...], ...}
```

In [89]:
years_awards = {}
with open("nobelPrizes.json", 'r') as f: 
    prizes = nobel["nobelPrizes"]
    for i in range(len(prizes)): 
        year = prizes[i]["awardYear"]
        category = prizes[i]["category"]["en"]
        if year not in years_awards:
            years_awards[year] = [category]
        else:
            years_awards[year].append(category)
    print(years_awards)

{'1901': ['Chemistry', 'Literature', 'Peace', 'Physics', 'Physiology or Medicine'], '1902': ['Chemistry', 'Literature', 'Peace', 'Physics', 'Physiology or Medicine'], '1903': ['Chemistry', 'Literature', 'Peace', 'Physics', 'Physiology or Medicine'], '1904': ['Chemistry', 'Literature', 'Peace', 'Physics', 'Physiology or Medicine'], '1905': ['Chemistry', 'Literature', 'Peace', 'Physics', 'Physiology or Medicine'], '1906': ['Chemistry', 'Literature', 'Peace', 'Physics', 'Physiology or Medicine'], '1907': ['Chemistry', 'Literature', 'Peace', 'Physics', 'Physiology or Medicine'], '1908': ['Chemistry', 'Literature', 'Peace', 'Physics', 'Physiology or Medicine'], '1909': ['Chemistry', 'Literature', 'Peace', 'Physics', 'Physiology or Medicine'], '1910': ['Chemistry', 'Literature', 'Peace', 'Physics', 'Physiology or Medicine'], '1911': ['Chemistry', 'Literature', 'Peace', 'Physics', 'Physiology or Medicine'], '1912': ['Chemistry', 'Literature', 'Peace', 'Physics', 'Physiology or Medicine'], '19

In [91]:
# Check
assert len(years_awards) == 25
assert years_awards["1903"] == ["Chemistry", "Literature", "Peace", "Physics", "Physiology or Medicine"]

How many years and how many awards does this dataset cover? (This is what the slide asks for.)

In [92]:
# YOUR CODE HERE
num_years = len(years_awards)

num_awards = 0
for categories in years_awards.values():
    num_awards += len(categories)

print("Years:", num_years)     # 25
print("Awards:", num_awards)   # 125

Years: 25
Awards: 125


## Step 4: Go further — who won?

**Predict:** before you build anything, look at the 1903 Physics prize. How many laureates do you think it has?

Each prize has a list of `laureates`. **Your turn:** build `winners`: a dictionary mapping each year to a list of `(category, name)` tuples.

Watch out:
- Some prizes have **more than one** laureate
- People have a `knownName`; organizations have an `orgName` instead. Use `.get()` to handle both.

In [112]:
winners = {}
for prize in nobel["nobelPrizes"]:
    year = prize["awardYear"]
    category = prize["category"]["en"]

    if year not in winners:
        winners[year] = []

    for laureate in prize.get("laureates", []):
        name_dict = laureate.get("knownName") or laureate.get("orgName")
        winners[year].append((category, name_dict["en"]))
print(winners)

{'1901': [('Chemistry', "Jacobus H. van 't Hoff"), ('Literature', 'Sully Prudhomme'), ('Peace', 'Henry Dunant'), ('Peace', 'Frédéric Passy'), ('Physics', 'Wilhelm Conrad Röntgen'), ('Physiology or Medicine', 'Emil von Behring')], '1902': [('Chemistry', 'Emil Fischer'), ('Literature', 'Theodor Mommsen'), ('Peace', 'Élie Ducommun'), ('Peace', 'Albert Gobat'), ('Physics', 'Hendrik A. Lorentz'), ('Physics', 'Pieter Zeeman'), ('Physiology or Medicine', 'Ronald Ross')], '1903': [('Chemistry', 'Svante Arrhenius'), ('Literature', 'Bjørnstjerne Bjørnson'), ('Peace', 'Randal Cremer'), ('Physics', 'Henri Becquerel'), ('Physics', 'Pierre Curie'), ('Physics', 'Marie Curie'), ('Physiology or Medicine', 'Niels Ryberg Finsen')], '1904': [('Chemistry', 'Sir William Ramsay'), ('Literature', 'Frédéric Mistral'), ('Literature', 'José Echegaray'), ('Peace', 'Institute of International Law'), ('Physics', 'Lord Rayleigh'), ('Physiology or Medicine', 'Ivan Pavlov')], '1905': [('Chemistry', 'Adolf von Baeyer')

Which prizes were shared? Print each prize that has more than one laureate.

In [114]:
new = {}
for data in winners.values():
    for key, name in data:
        if key not in new: 
            new[key] = 1
        else:
            new[key] += 1
    
print(new)

{'Chemistry': 22, 'Literature': 25, 'Peace': 27, 'Physics': 30, 'Physiology or Medicine': 23}


## Bonus: Export to JSON

Step 3 of the pattern, for JSON. Remember the four functions:
- `json.dump(data, f)` / `json.load(f)`: to and from a **file**
- `json.dumps(data)` / `json.loads(s)`: to and from a **string** (the `s` is for string)

Write `years_awards` to `years_awards.json` using `json.dump`. Then read it back in and check that you get the same dictionary.

Tip: `indent=2` makes the file readable.

In [115]:
# YOUR CODE HERE
# Write years_awards to a file
with open("years_awards.json", "w") as f:
    json.dump(years_awards, f, indent=2)

# Read it back in
with open("years_awards.json") as f:
    years_awards_back = json.load(f)

print(years_awards_back == years_awards)   # True


True


**Discuss:** we wrote `years_awards` to JSON and read it back. What would happen if the keys had been integers (`1901`) instead of strings (`"1901"`)? Predict, then try it below.

In [ ]:
int_keys = {1901: "Chemistry"}
round_trip = json.loads(json.dumps(int_keys))
print(round_trip)
print(round_trip == int_keys)

JSON object keys are **always strings**, so `1901` comes back as `"1901"` and the round trip isn't equal. That's one reason the Nobel API stores `awardYear` as a string.

### Wrap-up
With your partner: name the three steps of the pattern, and point to the cell in Part 1 *and* in Part 2 where each one happens.